# 04.2 — Baseline Comparison

Compares the engineered-feature pipeline selected in `04.1_model_training.ipynb` against reference baselines — majority-class, stratified-random, and TF-IDF text-only — and against several ways of combining TF-IDF with the engineered features.

This notebook is self-contained: it reloads the data and rebuilds the features and train/test split independently (same seed as `04.1_model_training.ipynb`, so results compare directly), rather than depending on `04.1_model_training.ipynb` having been run first.

## Setup

Reproduces the data load, `thread_id` grouping, feature matrix, and thread-grouped train/test split from `04.1_model_training.ipynb` (sections 8.1-8.3) — see that notebook for the reasoning behind each step.

In [1]:
import pandas as pd
import zipfile
import json

with zipfile.ZipFile('../results/03_cmv_comments_df.csv.zip') as z:
    comments_df = pd.read_csv(z.open(z.namelist()[0]))
comments_df = comments_df.copy()  # consolidate blocks - avoids a pandas PerformanceWarning on later single-column inserts

with open('../results/03_untrusted_features.json') as f:
    untrusted_features = json.load(f)

print(f'Loaded {len(comments_df)} rows, {comments_df.shape[1]} columns')
print(f'Untrusted features (used below only for the selected-pipeline reference point): {untrusted_features}')

Loaded 3971 rows, 237 columns
Untrusted features (used below only for the selected-pipeline reference point): ['sentiment_vader', 'tone_label', 'style_label', 'ethos_score', 'pathos_score', 'logos_score']


In [2]:
import hashlib

comments_df['thread_id'] = comments_df['original_post'].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

print(f'Unique threads : {comments_df["thread_id"].nunique()}')
print(f'Total comments : {len(comments_df)}')

Unique threads : 401
Total comments : 3971


In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder

# Comment-side, OP-side, and comment↔OP diff features from 03.1-03.3.
numeric_features = [
    'sentiment_llm', 'adj_adv_ratio', 'fk_grade_level', 'fk_reading_ease', 'evidence_count',
    'ethos_llm', 'pathos_llm', 'logos_llm',
    'sentiment_llm_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'evidence_count_op',
    'ethos_llm_op', 'pathos_llm_op', 'logos_llm_op',
    'sentiment_diff', 'ethos_diff', 'pathos_diff', 'logos_diff',
    'adj_adv_ratio_diff', 'fk_grade_level_diff', 'evidence_count_diff',
]
embedding_features = [c for c in comments_df.columns if c.startswith('embedding_')]
label = 'is_convincing'

categorical_features = ['tone_google', 'tone_google_op']
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features].fillna('unknown'))
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out(), index=comments_df.index)

comments_df['is_formal']    = (comments_df['style_google']    == 'Formal').astype(int)
comments_df['is_formal_op'] = (comments_df['style_google_op'] == 'Formal').astype(int)

match_features = ['sentiment_match', 'tone_match', 'style_match', 'ethos_match', 'pathos_match', 'logos_match']
match_df = comments_df[match_features].astype(int)

X = pd.concat([tone_df,
               comments_df[['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + numeric_features],
               match_df,
               comments_df[embedding_features]],
              axis=1)
y = comments_df[label]
groups = comments_df['thread_id']

print(f'Features : {X.shape[1]}  ({len(embedding_features)} of them PCA embedding components)')
print(f'Samples  : {X.shape[0]}')

Features : 228  (187 of them PCA embedding components)
Samples  : 3971


In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_threads = set(groups.iloc[train_idx])
test_threads  = set(groups.iloc[test_idx])
assert len(train_threads & test_threads) == 0, 'LEAKAGE: shared threads between train and test!'
print(f'Train: {len(train_idx)} comments from {len(train_threads)} threads')
print(f'Test : {len(test_idx)} comments from {len(test_threads)} threads')

Train: 3262 comments from 320 threads
Test : 709 comments from 81 threads


## Reference Point — the Selected Pipeline

`04.1_model_training.ipynb`'s validation grid (9 classifiers × 3 resampling strategies × 3 feature sets) selected **XGBoost + SMOTE oversampling on the `trusted+untrusted` feature set** by validation PR-AUC. Rather than re-running that full 27-combination grid here, this refits just that one pipeline on the same thread-grouped split, to get a live, directly comparable reference point for the baselines below.

In [5]:
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

untrusted_encoded = []
for col in untrusted_features:
    if pd.api.types.is_numeric_dtype(comments_df[col]):
        untrusted_encoded.append(comments_df[[col]].fillna(0))
    else:
        ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        encoded = ohe.fit_transform(comments_df[[col]].fillna('unknown'))
        untrusted_encoded.append(pd.DataFrame(encoded, columns=ohe.get_feature_names_out(), index=comments_df.index))

X_with_untrusted = pd.concat([X] + untrusted_encoded, axis=1)
X_train_selected = X_with_untrusted.iloc[train_idx]
X_test_selected  = X_with_untrusted.iloc[test_idx]

selected_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('sampler', SMOTE(random_state=42)),
    ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss')),
])
selected_pipe.fit(X_train_selected, y_train)
proba_selected = selected_pipe.predict_proba(X_test_selected)[:, 1]
pred_selected  = selected_pipe.predict(X_test_selected)

pr_auc_selected  = average_precision_score(y_test, proba_selected)
roc_auc_selected = roc_auc_score(y_test, proba_selected)
f1_selected       = f1_score(y_test, pred_selected, zero_division=0)

print('Selected pipeline (XGB + SMOTE oversampling, trusted+untrusted features):')
print(f'Test PR-AUC  : {pr_auc_selected:.4f}')
print(f'Test ROC-AUC : {roc_auc_selected:.4f}')
print(f'Test F1 @ 0.5: {f1_selected:.4f}  (untuned threshold - see 04.1_model_training.ipynb for the validation-tuned figure)')

Selected pipeline (XGB + SMOTE oversampling, trusted+untrusted features):
Test PR-AUC  : 0.3848
Test ROC-AUC : 0.6982
Test F1 @ 0.5: 0.2513  (untuned threshold - see 04.1_model_training.ipynb for the validation-tuned figure)


## 8.7 Baselines

The selected pipeline's test PR-AUC (above) is only meaningful next to some reference points. Six baselines are evaluated below, on the exact same thread-grouped train/test split as the real models:

- **Majority-class** — always predicts "not convincing" (the majority class). Not learned at all; makes concrete why accuracy is a misleading metric on this imbalanced dataset.
- **Stratified-random** — predicts randomly, matching the training set's class frequencies. This is the actual "no-skill" reference (equal to the positive rate) that PR-AUC is judged against.
- **TF-IDF + Logistic Regression** — a standard, minimal-effort NLP baseline: raw comment text only, none of the engineered features (no sentiment/tone/ethos-pathos-logos/embeddings/OP-relation features). Tests whether the `03.1`-`03.3` feature-engineering effort is actually earning its complexity.
- **TF-IDF + engineered features, combined (unscaled)** — TF-IDF stacked with the `trusted` engineered feature set (scalars + embeddings), no scaling on the dense block.
- **TF-IDF + engineered features, combined (fully scaled)** — the same combination, but with every dense engineered column, including the categorical/binary ones, passed through `RobustScaler` first, matching what the main model pipelines do in `04.1_model_training.ipynb`. Tests whether the unscaled version's underperformance was a scaling artifact.
- **TF-IDF + scalar features only, no embeddings (combined)** — TF-IDF stacked with just the ~41 hand-crafted scalar/categorical features, excluding the embedding columns. Tests whether combining TF-IDF with embeddings was adding redundant text signal rather than complementary signal.
- **TF-IDF + engineered features, combined (partial-scaled)** — the full trusted set again, but this time only the genuinely continuous columns (`numeric_features` + embeddings) are scaled; the categorical/binary columns (one-hot tone, `is_formal`/`_op`, `use_of_persuasive_lang`, the match flags) are left as-is. Tests whether scaling *everything*, including binary columns, was itself the problem in the previous scaled variant.

All are compared on PR-AUC and ROC-AUC (threshold-free, and the same primary/secondary metrics used throughout this project). F1 is also reported, but at each baseline's default 0.5 threshold rather than a validation-tuned one — unlike the selected pipeline, whose `04.1_model_training.ipynb` F1 used a threshold tuned on a validation set — so F1 isn't perfectly apples-to-apples between the two; PR-AUC is the fair comparison.

In [6]:
from sklearn.dummy import DummyClassifier

def bold_max(col):
    return ['font-weight: bold' if v == col.max() else '' for v in col]

baseline_results = {
    'Selected pipeline (XGB + SMOTE, trusted+untrusted)': {
        'PR-AUC': pr_auc_selected, 'ROC-AUC': roc_auc_selected, 'F1': f1_selected,
    },
}

for strategy, label in [('most_frequent', 'Majority-class'), ('stratified', 'Stratified-random')]:
    dummy = DummyClassifier(strategy=strategy, random_state=42)
    dummy.fit(X_train_selected, y_train)
    proba = dummy.predict_proba(X_test_selected)[:, 1]
    pred  = dummy.predict(X_test_selected)

    baseline_results[label] = {
        'PR-AUC':  average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'F1':      f1_score(y_test, pred, zero_division=0),
    }

pd.DataFrame(baseline_results).T.round(4)

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.3848,0.6982,0.2513
Majority-class,0.2031,0.5000,0.0000
Stratified-random,0.2013,0.4941,0.1699


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

text_train = comments_df['cleaned_final_comment'].iloc[train_idx].fillna('')
text_test  = comments_df['cleaned_final_comment'].iloc[test_idx].fillna('')

tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_text_train = tfidf.fit_transform(text_train)
X_text_test  = tfidf.transform(text_test)

tfidf_logreg = LogisticRegression(random_state=42, max_iter=1000)
tfidf_logreg.fit(X_text_train, y_train)
proba = tfidf_logreg.predict_proba(X_text_test)[:, 1]
pred  = tfidf_logreg.predict(X_text_test)

baseline_results['TF-IDF + LogReg'] = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

pd.DataFrame(baseline_results).T.round(4).style.apply(bold_max, axis=0)

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.384800,0.698200,0.251300
Majority-class,0.203100,0.500000,0.000000
Stratified-random,0.201300,0.494100,0.169900
TF-IDF + LogReg,0.511900,0.815000,0.123500


In [8]:
from scipy.sparse import hstack, csr_matrix

X_engineered_train = csr_matrix(X.iloc[train_idx].values)
X_engineered_test  = csr_matrix(X.iloc[test_idx].values)

X_combined_train = hstack([X_text_train, X_engineered_train])
X_combined_test  = hstack([X_text_test, X_engineered_test])

combined_logreg = LogisticRegression(random_state=42, max_iter=1000)
combined_logreg.fit(X_combined_train, y_train)
proba = combined_logreg.predict_proba(X_combined_test)[:, 1]
pred  = combined_logreg.predict(X_combined_test)

baseline_results['TF-IDF + engineered (combined)'] = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

pd.DataFrame(baseline_results).T.round(4).style.apply(bold_max, axis=0)

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.384800,0.698200,0.251300
Majority-class,0.203100,0.500000,0.000000
Stratified-random,0.201300,0.494100,0.169900
TF-IDF + LogReg,0.511900,0.815000,0.123500
TF-IDF + engineered (combined),0.455300,0.762100,0.244400


In [9]:
# Same combination as above, but scale the engineered block first - the main model
# pipelines in 04.1_model_training.ipynb all use RobustScaler for exactly this reason:
# without it, features on wildly different raw scales (fk_grade_level ~5-20, sentiment_llm
# -1 to 1, small PCA embedding floats, 0/1 booleans) can dominate the L2-regularized LogReg
# objective and drown out the already-normalized (unit L2 norm per document) TF-IDF signal.
engineered_scaler = RobustScaler()
X_engineered_train_scaled = engineered_scaler.fit_transform(X.iloc[train_idx])
X_engineered_test_scaled  = engineered_scaler.transform(X.iloc[test_idx])

X_combined_train_scaled = hstack([X_text_train, csr_matrix(X_engineered_train_scaled)])
X_combined_test_scaled  = hstack([X_text_test, csr_matrix(X_engineered_test_scaled)])

combined_logreg_scaled = LogisticRegression(random_state=42, max_iter=1000)
combined_logreg_scaled.fit(X_combined_train_scaled, y_train)
proba = combined_logreg_scaled.predict_proba(X_combined_test_scaled)[:, 1]
pred  = combined_logreg_scaled.predict(X_combined_test_scaled)

baseline_results['TF-IDF + engineered, scaled (combined)'] = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

pd.DataFrame(baseline_results).T.round(4).style.apply(bold_max, axis=0)

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.384800,0.698200,0.251300
Majority-class,0.203100,0.500000,0.000000
Stratified-random,0.201300,0.494100,0.169900
TF-IDF + LogReg,0.511900,0.815000,0.123500
TF-IDF + engineered (combined),0.455300,0.762100,0.244400
"TF-IDF + engineered, scaled (combined)",0.401800,0.714200,0.311900


In [10]:
# TF-IDF combined with the trusted engineered features, but excluding the embedding
# columns - combining TF-IDF with embeddings stacks two different representations of
# the same text, which may be adding redundant/conflicting signal rather than
# complementary signal. This isolates whether TF-IDF + the ~41 hand-crafted scalar/
# categorical features (sentiment, tone, ethos-pathos-logos, OP-relation diffs, etc.)
# - genuinely different information from the text itself - combines better than
# TF-IDF + the full trusted set (scalars + embeddings) above.
scalar_features = [c for c in X.columns if c not in embedding_features]

scalar_scaler = RobustScaler()
X_scalars_train_scaled = scalar_scaler.fit_transform(X.iloc[train_idx][scalar_features])
X_scalars_test_scaled  = scalar_scaler.transform(X.iloc[test_idx][scalar_features])

X_combined_scalars_train = hstack([X_text_train, csr_matrix(X_scalars_train_scaled)])
X_combined_scalars_test  = hstack([X_text_test, csr_matrix(X_scalars_test_scaled)])

combined_scalars_logreg = LogisticRegression(random_state=42, max_iter=1000)
combined_scalars_logreg.fit(X_combined_scalars_train, y_train)
proba = combined_scalars_logreg.predict_proba(X_combined_scalars_test)[:, 1]
pred  = combined_scalars_logreg.predict(X_combined_scalars_test)

baseline_results['TF-IDF + scalar features only, no embeddings (combined)'] = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

pd.DataFrame(baseline_results).T.round(4).style.apply(bold_max, axis=0)

,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.384800,0.698200,0.251300
Majority-class,0.203100,0.500000,0.000000
Stratified-random,0.201300,0.494100,0.169900
TF-IDF + LogReg,0.511900,0.815000,0.123500
TF-IDF + engineered (combined),0.455300,0.762100,0.244400
"TF-IDF + engineered, scaled (combined)",0.401800,0.714200,0.311900
"TF-IDF + scalar features only, no embeddings (combined)",0.465700,0.776700,0.193200


In [11]:
# Scale only the non-categorical (continuous) columns of the full trusted set -
# the categorical/binary ones (one-hot tone, is_formal/_op, use_of_persuasive_lang,
# match features) are already on a natural 0/1 scale. Scaling them too with
# RobustScaler (previous cell) is a likely reason that combination got worse instead
# of better: IQR-based scaling can blow up a skewed binary column's relative magnitude.
categorical_cols = list(tone_df.columns) + ['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + match_features
numeric_cols = [c for c in X.columns if c not in categorical_cols]
print(f'{len(categorical_cols)} categorical/binary columns, {len(numeric_cols)} numeric columns')

partial_scaler = RobustScaler()
X_numeric_train_scaled = partial_scaler.fit_transform(X.iloc[train_idx][numeric_cols])
X_numeric_test_scaled  = partial_scaler.transform(X.iloc[test_idx][numeric_cols])

X_engineered_train_partial = np.hstack([X_numeric_train_scaled, X.iloc[train_idx][categorical_cols].values])
X_engineered_test_partial  = np.hstack([X_numeric_test_scaled, X.iloc[test_idx][categorical_cols].values])

X_combined_train_partial = hstack([X_text_train, csr_matrix(X_engineered_train_partial)])
X_combined_test_partial  = hstack([X_text_test, csr_matrix(X_engineered_test_partial)])

combined_logreg_partial = LogisticRegression(random_state=42, max_iter=1000)
combined_logreg_partial.fit(X_combined_train_partial, y_train)
proba = combined_logreg_partial.predict_proba(X_combined_test_partial)[:, 1]
pred  = combined_logreg_partial.predict(X_combined_test_partial)

baseline_results['TF-IDF + engineered, partial-scaled (numeric only, combined)'] = {
    'PR-AUC':  average_precision_score(y_test, proba),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'F1':      f1_score(y_test, pred, zero_division=0),
}

pd.DataFrame(baseline_results).T.round(4).style.apply(bold_max, axis=0)

19 categorical/binary columns, 209 numeric columns


,PR-AUC,ROC-AUC,F1
"Selected pipeline (XGB + SMOTE, trusted+untrusted)",0.384800,0.698200,0.251300
Majority-class,0.203100,0.500000,0.000000
Stratified-random,0.201300,0.494100,0.169900
TF-IDF + LogReg,0.511900,0.815000,0.123500
TF-IDF + engineered (combined),0.455300,0.762100,0.244400
"TF-IDF + engineered, scaled (combined)",0.401800,0.714200,0.311900
"TF-IDF + scalar features only, no embeddings (combined)",0.465700,0.776700,0.193200
"TF-IDF + engineered, partial-scaled (numeric only, combined)",0.402800,0.714300,0.311900


**Majority-class** (PR-AUC 0.2031, F1 0.0) and **Stratified-random** (PR-AUC 0.2013, F1 0.1699) both land close to the no-skill baseline (the 0.161 positive rate — see `04.1_model_training.ipynb`). Majority-class's F1 of exactly 0 makes the accuracy point from that notebook's section 8.5 concrete: it would score 83.9% accuracy while never once correctly identifying a convincing comment.

**TF-IDF + Logistic Regression is still the strongest result in this table.** Its PR-AUC (0.5119) and ROC-AUC (0.8150) are both clearly *ahead* of the selected pipeline reproduced above (PR-AUC 0.3848, ROC-AUC 0.6982) — despite using none of the engineered features from `03.1`-`03.3`, just raw comment text. Its F1 (0.1235) is lower than the selected pipeline's validation-tuned F1 in `04.1_model_training.ipynb` (0.4046), but that's a threshold artifact (default 0.5 here vs. validation-tuned there), not a genuine reversal — on the threshold-free metrics this notebook treats as primary and secondary, TF-IDF+LogReg wins clearly.

**Scaling the engineered block consistently hurts the combination, and it isn't about which columns get scaled.** Four combined variants were tried:
- `TF-IDF + scalar features only, no embeddings` (0.4657) > `TF-IDF + engineered, unscaled` (0.4553) > `TF-IDF + engineered, partial-scaled (numeric only)` (0.4028) ≈ `TF-IDF + engineered, fully scaled` (0.4018).

The partial-scaled and fully-scaled variants land within 0.001 of each other — scaling only the 209 continuous columns and leaving the 19 categorical/binary columns untouched performs essentially identically to scaling everything. That rules out the original hypothesis (that inflating skewed binary columns was the specific problem) and points to a simpler explanation: `TfidfVectorizer` L2-normalizes each document's vector by default, so the sparse text block already sits at a small, consistent scale. The *unscaled* engineered block (mostly bounded values — sentiment in [-1,1], ethos/pathos/logos in [0,1], small PCA embedding floats) is naturally "quiet" next to that, so it nudges the fit without overpowering it — which is why unscaled combinations come closest to TF-IDF alone. `RobustScaler` normalizes every numeric column to a comparable spread, which makes the dense block collectively *louder* across all 209 columns, not better-behaved — and that competes with the text signal in the shared L2-regularized objective rather than complementing it, regardless of whether the binary columns are included in the scaling.

Dropping the embeddings and keeping just the ~41 scalar features (unscaled apart from the earlier combined-alone test) remains the best combined attempt (0.4657) — consistent with the `embeddings-only` finding in `04.1_model_training.ipynb`'s validation grid, that combining TF-IDF with embeddings mostly adds a second, redundant representation of the same text rather than complementary signal. But even that doesn't beat TF-IDF by itself (0.5119).

Taken together: the gap here isn't "engineered features vs. no features," and it isn't a scaling bug either — it's that a raw, uncompressed bag-of-words representation currently outperforms every naive way of combining it with the engineered features tried so far, and forcing all continuous features onto the same scale as an already-normalized sparse text matrix makes things worse, not better. It doesn't mean the engineered features are worthless — the selected pipeline still clearly beats the majority/random baselines, and the untrusted-feature comparison in `04.1_model_training.ipynb` showed real, if modest, per-feature signal — but naive concatenation into a single untuned LogReg isn't the right way to combine them. A stacked/ensembled combination of separately-tuned models, or a staged ablation (surface-only → +sentiment/tone → +embeddings → +strategy scores) as a follow-up piece of work, are more promising next steps than further tweaking this quick diagnostic.

## 8.8 Feature-Family Ablation

TF-IDF alone beats every naive combination with the *full* engineered feature set (previous section). That doesn't rule out a smaller subset helping — a genuinely useful family could still be diluted or outweighed by noisier ones when all ~41 scalar/categorical features are dumped in together.

To check, the engineered features are grouped into eight families and added to `TF-IDF + LogReg` one at a time:

- **sentiment**, **tone/style**, **readability**, **evidence use**, **ethos/pathos/logos** — the comment's own linguistic properties (comment-side only, no `_op`/`_diff`/`_match` columns)
- **OP features** — the same measurements computed on the original post instead of the comment
- **diff/match** — the comment↔OP relational features (numeric diffs and boolean match flags) from `03.3`
- **embeddings** — the 187 PCA-compressed document embedding components

Two views are computed: **solo** (`TF-IDF + family X` alone, isolating that family's standalone lift or drag) and **cumulative** (`TF-IDF + families[:i+1]`, families added in the order above, surfacing interaction effects that solo addition can't catch). Both reuse the unscaled combination style from the previous section, since scaling was already shown there to hurt rather than help.

In [12]:
op_tone_cols = [c for c in tone_df.columns if c.startswith('tone_google_op_')]
comment_tone_cols = [c for c in tone_df.columns if c not in op_tone_cols]

FEATURE_FAMILIES = {
    'sentiment':          ['sentiment_llm'],
    'tone/style':         comment_tone_cols + ['is_formal', 'adj_adv_ratio'],
    'readability':        ['fk_grade_level', 'fk_reading_ease'],
    'evidence use':       ['evidence_count', 'use_of_persuasive_lang'],
    'ethos/pathos/logos': ['ethos_llm', 'pathos_llm', 'logos_llm'],
    'OP features':        op_tone_cols + ['is_formal_op', 'adj_adv_ratio_op', 'fk_grade_level_op',
                                           'evidence_count_op', 'sentiment_llm_op', 'ethos_llm_op',
                                           'pathos_llm_op', 'logos_llm_op'],
    'diff/match':         ['sentiment_diff', 'ethos_diff', 'pathos_diff', 'logos_diff',
                            'adj_adv_ratio_diff', 'fk_grade_level_diff', 'evidence_count_diff'] + match_features,
    'embeddings':         embedding_features,
}

# The families must exactly partition X's columns - no column left out, none double-counted -
# otherwise the cumulative view below wouldn't actually reach the full engineered feature set.
all_family_cols = [c for cols in FEATURE_FAMILIES.values() for c in cols]
assert len(all_family_cols) == len(set(all_family_cols)), 'a column appears in more than one family'
assert set(all_family_cols) == set(X.columns), 'family columns do not match X columns exactly'

for name, cols in FEATURE_FAMILIES.items():
    print(f'{name:20s}: {len(cols):3d} columns')

sentiment           :   1 columns
tone/style          :   7 columns
readability         :   2 columns
evidence use        :   2 columns
ethos/pathos/logos  :   3 columns
OP features         :  13 columns
diff/match          :  13 columns
embeddings          : 187 columns


In [13]:
def evaluate_tfidf_plus_features(feature_cols, source_df=None):
    """Fit TF-IDF + LogReg with the given engineered columns stacked on, unscaled (matching
    the best-performing combination style from the Baselines section above). source_df lets
    the untrusted-feature follow-up below pull columns that live in X_with_untrusted, not X."""
    if source_df is None:
        source_df = X
    X_extra_train = csr_matrix(source_df.iloc[train_idx][feature_cols].values)
    X_extra_test  = csr_matrix(source_df.iloc[test_idx][feature_cols].values)
    X_train_combo = hstack([X_text_train, X_extra_train])
    X_test_combo  = hstack([X_text_test, X_extra_test])

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_combo, y_train)
    proba = model.predict_proba(X_test_combo)[:, 1]
    pred  = model.predict(X_test_combo)

    return {
        'PR-AUC':  average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'F1':      f1_score(y_test, pred, zero_division=0),
    }


tfidf_alone = baseline_results['TF-IDF + LogReg']

solo_results = {'TF-IDF alone': tfidf_alone}
for family, cols in FEATURE_FAMILIES.items():
    solo_results[f'+ {family}'] = evaluate_tfidf_plus_features(cols)

solo_df = pd.DataFrame(solo_results).T.round(4)
solo_df['Delta PR-AUC vs TF-IDF alone'] = (solo_df['PR-AUC'] - tfidf_alone['PR-AUC']).round(4)
solo_df.sort_values('PR-AUC', ascending=False)

,PR-AUC,ROC-AUC,F1,Delta PR-AUC vs TF-IDF alone
+ ethos/pathos/logos,0.5221,0.8151,0.1595,0.0102
+ diff/match,0.5215,0.8079,0.2024,0.0096
+ sentiment,0.5130,0.8162,0.1350,0.0011
TF-IDF alone,0.5119,0.8150,0.1235,0.0000
+ tone/style,0.5106,0.8181,0.1775,-0.0013
+ evidence use,0.5103,0.8132,0.1350,-0.0016
+ embeddings,0.5097,0.7926,0.2414,-0.0022
+ readability,0.4987,0.8108,0.1333,-0.0132
+ OP features,0.4697,0.7784,0.1350,-0.0422


In [14]:
cumulative_results = {'TF-IDF alone': tfidf_alone}
cumulative_cols = []
for family, cols in FEATURE_FAMILIES.items():
    cumulative_cols = cumulative_cols + cols
    cumulative_results[f'+ {family}'] = evaluate_tfidf_plus_features(cumulative_cols)

cumulative_df = pd.DataFrame(cumulative_results).T.round(4)
cumulative_df['Delta PR-AUC vs previous row'] = cumulative_df['PR-AUC'].diff().round(4)
cumulative_df

,PR-AUC,ROC-AUC,F1,Delta PR-AUC vs previous row
TF-IDF alone,0.5119,0.8150,0.1235,NaN
+ sentiment,0.5130,0.8162,0.1350,0.0011
+ tone/style,0.5114,0.8177,0.1765,-0.0016
+ readability,0.5083,0.8145,0.1882,-0.0031
+ evidence use,0.5068,0.8127,0.1786,-0.0015
+ ethos/pathos/logos,0.5108,0.8032,0.1916,0.0040
+ OP features,0.4654,0.7755,0.1932,-0.0454
+ diff/match,0.4631,0.7762,0.1932,-0.0023
+ embeddings,0.4559,0.7625,0.2444,-0.0072


In [15]:
# Solo results above flag exactly two families with a real (non-negligible) standalone lift:
# ethos/pathos/logos (+0.0102) and diff/match (+0.0096); sentiment is roughly a wash (+0.0011).
# The cumulative view adds families in a fixed order and can't isolate whether *these three*
# specifically compound when combined alone, skipping the families that hurt (OP features,
# readability, tone/style, evidence use, embeddings) - so that combination is tested directly.
best_subset_cols = FEATURE_FAMILIES['sentiment'] + FEATURE_FAMILIES['ethos/pathos/logos'] + FEATURE_FAMILIES['diff/match']
best_subset_result = evaluate_tfidf_plus_features(best_subset_cols)

print(f'TF-IDF + {{sentiment, ethos/pathos/logos, diff/match}} ({len(best_subset_cols)} columns):')
print(f'PR-AUC : {best_subset_result["PR-AUC"]:.4f}  (TF-IDF alone: {tfidf_alone["PR-AUC"]:.4f})')
print(f'ROC-AUC: {best_subset_result["ROC-AUC"]:.4f}  (TF-IDF alone: {tfidf_alone["ROC-AUC"]:.4f})')
print(f'F1     : {best_subset_result["F1"]:.4f}  (TF-IDF alone: {tfidf_alone["F1"]:.4f})')

TF-IDF + {sentiment, ethos/pathos/logos, diff/match} (17 columns):
PR-AUC : 0.5124  (TF-IDF alone: 0.5119)
ROC-AUC: 0.8046  (TF-IDF alone: 0.8150)
F1     : 0.1916  (TF-IDF alone: 0.1235)


**Two families show a small but real standalone lift over TF-IDF alone**: `ethos/pathos/logos` (+0.0102 PR-AUC) and `diff/match` (+0.0096) both beat the 0.5119 baseline on their own. `sentiment` is roughly a wash (+0.0011). Every other family is flat-to-negative in isolation — `readability` costs -0.0132, and `OP features` is clearly the worst, at -0.0422, more than four times the next-largest drag.

**But the gains don't compound.** Stacking just the two families with a real standalone lift — `TF-IDF + {sentiment, ethos/pathos/logos, diff/match}` — scores 0.5124 PR-AUC, within noise of TF-IDF alone (0.5119), and its ROC-AUC (0.8046) is actually *lower* than TF-IDF alone (0.8150). Whatever small signal each family contributes on its own overlaps rather than adds when combined into the same L2-regularized LogReg fit.

**The cumulative view shows where the full stack collapses.** By the row where `OP features` enters, PR-AUC drops from 0.5108 to 0.4654 (-0.0454) — close to `OP features`' own solo penalty (-0.0422) — and neither `diff/match`'s later solo lift nor the earlier gains recover from that drop. The final row (all 8 families = the full trusted feature set) lands at 0.4559 PR-AUC, matching — within floating-point noise from a different column order feeding the same solver — the `TF-IDF + engineered (combined)` figure of 0.4553 from the Baselines section above, a useful cross-check that this ablation correctly reconstructs the same full-feature result via a different path.

**Bottom line for the "does any single family help" question**: no family, or even the best-looking pair combined, clears TF-IDF alone by a meaningful margin. `ethos/pathos/logos` and `diff/match` are the only families worth revisiting in future feature work (e.g. a properly regularized/stacked combination rather than raw concatenation), but they don't explain — let alone reverse — why the full engineered feature set underperforms TF-IDF. `OP features` stand out as actively harmful, both alone and as the point where the cumulative stack collapses — worth a closer look (noisier annotations on the original-post side, or genuinely weaker signal there) before trusting them in any future model.

### Follow-ups: dropping `OP features`, and adding the untrusted features

Two more targeted combinations, motivated directly by the results above:

- **All families except `OP features`** — the cumulative table's row just before `OP features` enters (`+ ethos/pathos/logos`, i.e. the five other comment-side families) already sits at 0.5108, only 0.0011 below TF-IDF alone. This checks whether adding the two remaining non-OP families (`diff/match`, `embeddings`) on top of that holds up, now that the one clearly harmful family is left out entirely.
- **The (technically) untrusted features** — `sentiment_vader`, `tone_label`, `style_label`, `ethos_score`, `pathos_score`, `logos_score` were flagged in `03.1`/`03.2` as not reliable enough for the main model, but `04.1_model_training.ipynb`'s validation grid found they still help some model/resampling combinations (`XGB (over)` gains +0.063 PR-AUC from them). Worth checking whether they help here too — both alone, and stacked onto the best non-OP combination above.

In [16]:
non_op_cols = [c for family, cols in FEATURE_FAMILIES.items() if family != 'OP features' for c in cols]
non_op_result = evaluate_tfidf_plus_features(non_op_cols)

print(f'TF-IDF + all families except OP features ({len(non_op_cols)} columns):')
print(f'PR-AUC : {non_op_result["PR-AUC"]:.4f}  (TF-IDF alone: {tfidf_alone["PR-AUC"]:.4f})')
print(f'ROC-AUC: {non_op_result["ROC-AUC"]:.4f}  (TF-IDF alone: {tfidf_alone["ROC-AUC"]:.4f})')
print(f'F1     : {non_op_result["F1"]:.4f}  (TF-IDF alone: {tfidf_alone["F1"]:.4f})')

TF-IDF + all families except OP features (215 columns):
PR-AUC : 0.4830  (TF-IDF alone: 0.5119)
ROC-AUC: 0.7778  (TF-IDF alone: 0.8150)
F1     : 0.2373  (TF-IDF alone: 0.1235)


In [17]:
# untrusted_encoded/X_with_untrusted were already built in the Reference Point section above,
# for the selected engineered-feature pipeline - reused here rather than re-encoding. Both
# calls pass source_df=X_with_untrusted since untrusted_cols only exist there, not in X.
untrusted_cols = [c for c in X_with_untrusted.columns if c not in X.columns]

untrusted_solo_result        = evaluate_tfidf_plus_features(untrusted_cols, source_df=X_with_untrusted)
non_op_plus_untrusted_result = evaluate_tfidf_plus_features(non_op_cols + untrusted_cols, source_df=X_with_untrusted)

untrusted_comparison = pd.DataFrame({
    'TF-IDF alone':                        tfidf_alone,
    'TF-IDF + untrusted features':         untrusted_solo_result,
    'TF-IDF + all except OP':              non_op_result,
    'TF-IDF + all except OP + untrusted':  non_op_plus_untrusted_result,
}).T.round(4)
untrusted_comparison

,PR-AUC,ROC-AUC,F1
TF-IDF alone,0.5119,0.8150,0.1235
TF-IDF + untrusted features,0.5152,0.8067,0.1576
TF-IDF + all except OP,0.4830,0.7778,0.2373
TF-IDF + all except OP + untrusted,0.4907,0.7769,0.2386


**Dropping `OP features` doesn't rescue the full stack.** `TF-IDF + all families except OP features` (215 columns) scores 0.4830 PR-AUC — *worse* than TF-IDF alone (0.5119) by -0.0289, and also worse than the cumulative row that stopped right before `OP features` entered (0.5108, but that row only carried 5 families). The two families left to add at that point — `diff/match` and `embeddings` — looked individually harmless-to-positive in isolation (diff/match: +0.0096 solo, embeddings: -0.0022 solo), but together with the other five they compound into a real net loss. So `OP features` was the single biggest problem, but not the *only* one: the accumulated small drags from `tone/style`, `readability`, `evidence use`, and `embeddings` are enough on their own to pull the combination below TF-IDF alone once they're all stacked together, even with the worst offender removed.

**The untrusted features give a small, genuine solo lift** — `TF-IDF + untrusted features` reaches 0.5152 PR-AUC, +0.0033 over TF-IDF alone, in the same modest range as `sentiment`'s solo lift (+0.0011) and smaller than `ethos/pathos/logos` (+0.0102) or `diff/match` (+0.0096). This matches what `04.1_model_training.ipynb`'s validation grid already found for the pure engineered-feature model: the untrusted features aren't uniformly harmful, and can add a little signal on top of already-decent features. ROC-AUC moves the other way here (0.8067 vs. 0.8150), so as with the other families, "small positive PR-AUC lift, roughly flat-to-negative ROC-AUC" is the recurring pattern, not a clean win.

**Stacking both together only partially recovers.** `TF-IDF + all except OP + untrusted` reaches 0.4907 — better than dropping OP features alone (0.4830, +0.0077 from adding the untrusted features), but still -0.0212 below TF-IDF alone. The untrusted features' modest solo lift doesn't compound any better than the trusted families' did earlier in this section.

**Overall**: across every combination tried in this ablation — single families, the best-looking pair, everything except `OP features`, and now the untrusted features on top of that — nothing has closed the gap to TF-IDF alone. The consistent pattern is small, real, individually-measurable lifts from several sources (`ethos/pathos/logos`, `diff/match`, `sentiment`, the untrusted features) that don't survive being combined into a single shared LogReg fit. That's the strongest case yet for trying option 1 — a stacked/ensembled combination of separately-tuned models — since simple concatenation, with or without the worst-offending families, keeps landing in the same place.

## 8.9 TF-IDF Interpretability

`TF-IDF + LogReg` (no engineered features at all) is the strongest model found so far. That's worth treating with some suspicion rather than taken at face value — a bag-of-words model can win by picking up genuine rhetorical signal (hedges, discourse markers, politeness), or it can win by memorizing topic- or thread-specific vocabulary that happens to correlate with `is_convincing` in this particular dataset without generalizing.

`tfidf_logreg` (fit earlier in the Baselines section) is a plain linear model over TF-IDF features, so its coefficients are directly interpretable: each one is the (L2-regularized) log-odds weight for one token. The tokens with the most extreme coefficients in each direction are inspected below.

In [18]:
n_top = 20

coef_df = pd.DataFrame({
    'token':       tfidf.get_feature_names_out(),
    'coefficient': tfidf_logreg.coef_[0],
})

top_convincing     = coef_df.sort_values('coefficient', ascending=False).head(n_top).reset_index(drop=True)
top_not_convincing = coef_df.sort_values('coefficient').head(n_top).reset_index(drop=True)

token_comparison = pd.concat(
    [top_convincing.add_suffix(' (→ convincing)'), top_not_convincing.add_suffix(' (→ not convincing)')],
    axis=1,
)
token_comparison

,token (→ convincing),coefficient (→ convincing),token (→ not convincing),coefficient (→ not convincing)
0,and,2.473082,op,-0.967558
1,the,2.349566,comment,-0.785318
2,your,1.883098,im,-0.760344
3,on,1.746659,thank,-0.706694
4,edit,1.718478,100,-0.696787
5,of,1.580126,white,-0.692626
6,to,1.461363,thanks,-0.689103
7,you,1.449624,well,-0.685701
8,is,1.273756,looking,-0.678066
9,people,1.245915,capitalism,-0.649210


**Function words dominate the "convincing" side.** Twelve of the top 20 tokens pushing toward `is_convincing` are common stopwords — `and`, `the`, `your`, `on`, `of`, `to`, `you`, `is`, `their`, `would`, `in`, `are` — rather than anything specifically rhetorical. `TfidfVectorizer` wasn't configured to strip stopwords, so this is plausible: longer, more elaborated comments simply contain more function words in absolute terms, and comment length/elaboration is a known proxy for argument effort on CMV. That means part of what looks like "text beats engineered features" may really be "longer comments beat engineered features" — worth checking directly (e.g. correlating raw comment length with `is_convincing`) as a follow-up before trusting this as rhetorical signal.

**`edit` is a top positive token (rank 5, coefficient 1.72) and worth flagging as a possible artifact rather than genuine rhetorical signal.** CMV commenters sometimes edit a comment after posting it — to add a source, respond to follow-up questions, or acknowledge a delta — and any of those reasons could make `edit` correlate with `is_convincing` for a reason unrelated to the original argument's persuasiveness. Worth a manual read-through of a sample of `edit`-containing comments before trusting it as a stable feature.

**Several top tokens are thread/topic-specific rather than general-purpose**: `israel`, `gun`, `trans`, `platform` on the convincing side, and `capitalism`, `white`, `profit` on the not-convincing side, point to particular CMV threads about specific hot-button topics rather than transferable argument style. Thread-grouped splitting (`04.1_model_training.ipynb` sections 8.2-8.3) stops the *same* thread from leaking across train/test, but it doesn't stop the model from learning "threads about X tend to have more/fewer deltas" as a shortcut — which wouldn't generalize to a new topic.

**Some "not convincing" tokens are mildly counter-intuitive**: `thank`/`thanks`, `agree`, `exactly`, `sure` all *reduce* predicted convincingness, despite reading as polite or affirming. A plausible reading: comments that agree with or thank another commenter — rather than directly challenging OP's stated view — are less likely to be the specific comment that earns a delta, since deltas reward changing a view, not affirming it. `op` itself is the single most negative token — comments that explicitly reference "OP" in the third person read as meta-commentary about the discussion rather than a direct argument, which fits the same pattern.

Overall: TF-IDF's advantage over the engineered features is real on this test set, but at least part of it looks like it's coming from comment length/elaboration and topic-specific vocabulary rather than purely from generalizable rhetorical strategy. A length-controlled comparison, or evaluating on topics held out entirely from training, would be a stronger next check before concluding that raw text beats engineered features as a general rule rather than as an artifact of this particular dataset.

## 8.10 Stacked Ensemble (TF-IDF + Engineered Features, Blended)

Every naive concatenation tried above — the full engineered set, individual families, the best-looking pair, everything except `OP features`, even with the untrusted features added — landed at or below TF-IDF alone. All of those share the same failure mode: cramming a large, well-behaved TF-IDF block and a much smaller stack of differently-scaled engineered features into one shared, L2-regularized objective, where they compete rather than complement each other.

This section tries a different combination strategy: train TF-IDF+LR and the selected engineered pipeline (`selected_pipe`: XGB + SMOTE oversampling on `trusted+untrusted`) completely separately, each on its own natural representation, then blend their predicted probabilities. The blend weight is chosen on a held-out validation split — the same thread-grouped inner-train/validation split `04.1_model_training.ipynb` already uses for threshold tuning — rather than a fresh k-fold cross-validation scheme, since fitting one or two parameters doesn't need more data than that split already provides. Two combiners are tried and the better one (on validation) is carried to the test set: a single fixed blend weight, and a small logistic-regression meta-model over both models' probabilities.

In [19]:
# Same thread-grouped inner-train/validation split as 04.1_model_training.ipynb's threshold
# tuning (same random_state=0), reproduced here rather than depending on that notebook having
# been run - inner_idx/val_idx are positions relative to train_idx, not absolute row positions.
train_groups = groups.iloc[train_idx].reset_index(drop=True)
y_train_r    = y_train.reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
inner_idx, val_idx = next(gss_val.split(np.zeros((len(train_idx), 1)), y_train_r, groups=train_groups))

y_inner = y_train_r.iloc[inner_idx]
y_val   = y_train_r.iloc[val_idx]

inner_threads = set(train_groups.iloc[inner_idx])
val_threads   = set(train_groups.iloc[val_idx])
assert len(inner_threads & val_threads) == 0
print(f'Inner train: {len(inner_idx)} comments from {len(inner_threads)} threads')
print(f'Validation : {len(val_idx)} comments from {len(val_threads)} threads')

Inner train: 2629 comments from 256 threads
Validation : 633 comments from 64 threads


In [20]:
# TF-IDF + LogReg, fit on inner only (not the full training set) so its validation-set
# predictions are genuinely out-of-sample - needed to pick a blend weight honestly.
text_inner = text_train.iloc[inner_idx]
text_val   = text_train.iloc[val_idx]

tfidf_inner = TfidfVectorizer(max_features=5000, min_df=2)
X_text_inner_fit = tfidf_inner.fit_transform(text_inner)
X_text_val_fit   = tfidf_inner.transform(text_val)

tfidf_logreg_inner = LogisticRegression(random_state=42, max_iter=1000)
tfidf_logreg_inner.fit(X_text_inner_fit, y_inner)
proba_tfidf_val = tfidf_logreg_inner.predict_proba(X_text_val_fit)[:, 1]

# Engineered pipeline - same architecture as selected_pipe above - fit on inner only.
X_train_selected_inner = X_train_selected.iloc[inner_idx]
X_train_selected_val   = X_train_selected.iloc[val_idx]

eng_pipe_inner = Pipeline([
    ('scaler', RobustScaler()),
    ('sampler', SMOTE(random_state=42)),
    ('clf', xgb.XGBClassifier(random_state=42, eval_metric='logloss')),
])
eng_pipe_inner.fit(X_train_selected_inner, y_inner)
proba_eng_val = eng_pipe_inner.predict_proba(X_train_selected_val)[:, 1]

print(f'TF-IDF (inner-fit) val PR-AUC     : {average_precision_score(y_val, proba_tfidf_val):.4f}')
print(f'Engineered (inner-fit) val PR-AUC : {average_precision_score(y_val, proba_eng_val):.4f}')

TF-IDF (inner-fit) val PR-AUC     : 0.3971
Engineered (inner-fit) val PR-AUC : 0.3063


In [21]:
alphas = np.linspace(0, 1, 101)
alpha_scores = [
    average_precision_score(y_val, a * proba_tfidf_val + (1 - a) * proba_eng_val)
    for a in alphas
]
best_alpha = alphas[int(np.argmax(alpha_scores))]
best_alpha_pr_auc = max(alpha_scores)

print(f'Best blend weight (alpha, TF-IDF share): {best_alpha:.2f}')
print(f'Validation PR-AUC at that weight: {best_alpha_pr_auc:.4f}')
print(f'  (TF-IDF alone on val: {average_precision_score(y_val, proba_tfidf_val):.4f},'
      f' engineered alone on val: {average_precision_score(y_val, proba_eng_val):.4f})')

Best blend weight (alpha, TF-IDF share): 0.83
Validation PR-AUC at that weight: 0.4099
  (TF-IDF alone on val: 0.3971, engineered alone on val: 0.3063)


In [22]:
meta_X_val = np.column_stack([proba_tfidf_val, proba_eng_val])
meta_logreg = LogisticRegression(random_state=42)
meta_logreg.fit(meta_X_val, y_val)
meta_proba_val = meta_logreg.predict_proba(meta_X_val)[:, 1]
meta_pr_auc_val = average_precision_score(y_val, meta_proba_val)

# Both the alpha search and this meta-model were fit and scored on the same validation set -
# a fair head-to-head, since both have equally little capacity to overfit it (1 parameter vs.
# 2 coefficients + intercept). Test remains untouched by either choice made here.
use_meta_logreg = meta_pr_auc_val > best_alpha_pr_auc

print(f'Learned LR meta-combiner val PR-AUC: {meta_pr_auc_val:.4f}  (fixed-alpha blend: {best_alpha_pr_auc:.4f})')
print(f'Selected combiner for test evaluation: {"learned LR meta-model" if use_meta_logreg else f"fixed blend (alpha={best_alpha:.2f})"}')

Learned LR meta-combiner val PR-AUC: 0.4020  (fixed-alpha blend: 0.4099)
Selected combiner for test evaluation: fixed blend (alpha=0.83)


In [23]:
# Base-model test-set probabilities, from the pipelines already fit on the FULL training set
# above (tfidf_logreg in the Baselines section, selected_pipe in the Reference Point section) -
# not the inner-only fits used above, which existed only to pick the combiner honestly.
proba_tfidf_test = tfidf_logreg.predict_proba(X_text_test)[:, 1]
proba_eng_test   = selected_pipe.predict_proba(X_test_selected)[:, 1]

if use_meta_logreg:
    meta_X_test = np.column_stack([proba_tfidf_test, proba_eng_test])
    proba_stacked_test = meta_logreg.predict_proba(meta_X_test)[:, 1]
else:
    proba_stacked_test = best_alpha * proba_tfidf_test + (1 - best_alpha) * proba_eng_test

pred_stacked_test = (proba_stacked_test >= 0.5).astype(int)

stacked_result = {
    'PR-AUC':  average_precision_score(y_test, proba_stacked_test),
    'ROC-AUC': roc_auc_score(y_test, proba_stacked_test),
    'F1':      f1_score(y_test, pred_stacked_test, zero_division=0),
}

stacking_comparison = pd.DataFrame({
    'TF-IDF alone':                 tfidf_alone,
    'Selected engineered pipeline': {'PR-AUC': pr_auc_selected, 'ROC-AUC': roc_auc_selected, 'F1': f1_selected},
    'Stacked (blended)':            stacked_result,
}).T.round(4)
stacking_comparison

,PR-AUC,ROC-AUC,F1
TF-IDF alone,0.5119,0.8150,0.1235
Selected engineered pipeline,0.3848,0.6982,0.2513
Stacked (blended),0.5152,0.8093,0.1366


In [24]:
def bootstrap_ci(y_true, proba, metric_fn, n_boot=1000, seed=42):
    """95% CI for a metric_fn(y_true_sample, proba_sample) via resampling with replacement."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    proba  = np.asarray(proba)
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, pb = y_true[idx], proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        scores.append(metric_fn(yt, pb))
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

ci_tfidf   = bootstrap_ci(y_test, proba_tfidf_test, average_precision_score)
ci_eng     = bootstrap_ci(y_test, proba_eng_test, average_precision_score)
ci_stacked = bootstrap_ci(y_test, proba_stacked_test, average_precision_score)

pr_auc_ci_comparison = pd.DataFrame({
    'PR-AUC':        [tfidf_alone['PR-AUC'], pr_auc_selected, stacked_result['PR-AUC']],
    'PR-AUC 95% CI': [ci_tfidf, ci_eng, ci_stacked],
}, index=['TF-IDF alone', 'Selected engineered pipeline', 'Stacked (blended)'])
pr_auc_ci_comparison

,PR-AUC,PR-AUC 95% CI
TF-IDF alone,0.511861,"(0.4296995810391229, 0.6031731987916423)"
Selected engineered pipeline,0.384771,"(0.3114300838772552, 0.4629195832057603)"
Stacked (blended),0.515226,"(0.433401803099745, 0.6049344980448894)"


**The fixed-weight blend beat the learned meta-model on validation** — 0.4099 PR-AUC vs. 0.4020 — so it's the one carried to the test set, at `alpha = 0.83` (83% TF-IDF, 17% engineered). That weight itself is informative: it's close to what the two models' relative validation strength would suggest (TF-IDF alone: 0.3971, engineered alone: 0.3063), rather than an even split — the search correctly leaned hard toward the stronger model instead of averaging them naively.

**On the test set, stacking edges out TF-IDF alone** — PR-AUC 0.5152 vs. 0.5119 (+0.0033), F1 0.1366 vs. 0.1235 (+0.0131) — while ROC-AUC is essentially flat (0.8093 vs. 0.8150). This is a genuinely different outcome from every naive-concatenation attempt above, all of which landed *below* TF-IDF alone (the best of those, `TF-IDF + scalar features only`, still trailed by -0.0462 PR-AUC). Separating the two models and blending their probabilities, rather than forcing both representations through one shared objective, is the first approach in this notebook that doesn't cost PR-AUC relative to TF-IDF alone.

**But the bootstrap CIs make clear this isn't a confident win.** TF-IDF alone's PR-AUC 95% CI is [0.430, 0.603]; the stacked blend's is [0.433, 0.605] — almost the same interval, just shifted by the same +0.003 as the point estimate. A gain this small, on a test set this size (709 comments, 81 threads), is well within the noise floor already established elsewhere in this project (recall the leakage comparison in `04.1_model_training.ipynb` couldn't distinguish two PR-AUC estimates 0.03 apart with overlapping CIs).

**Bottom line**: stacking validates the underlying hypothesis — the engineered features *can* be combined with TF-IDF without hurting it, once the two representations stop competing inside the same regularized fit — but it doesn't produce a result clearly better than using TF-IDF alone. Given how heavily the optimal blend leans toward TF-IDF (`alpha = 0.83`) and how weak the engineered pipeline is standalone (PR-AUC 0.3848 vs. TF-IDF's 0.5119), that's not surprising: there's a real ceiling on how much a much weaker model can lift a much stronger one through blending alone, however cleanly it's combined. A stronger engineered-features model (rather than a better combination strategy) looks like the more promising next lever — which points back to the original plan of trying rhetorical-strategy scores (MS-PS framework) or task-focused embeddings (e5) as **new** signal, rather than further recombination of what's already been tried.